In [6]:
# Baseline "aktuelles Regelwerk"
import sys
from pathlib import Path
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))

from pathlib import Path

import pandas as pd
import numpy as np

def simple_baseline_metrics(df: pd.DataFrame, alpha: float = 0.05, ensure_unique_tx: bool = True):
    """
    Compute baseline KPIs without any modeling.

    Returns a dict with:
      - success_rate
      - avg_cost
      - avg_fee_by_success (dict for success==0/1)
      - business_score (success_rate - alpha * avg_cost)
    """
    s = df.copy()

    # Ensure proper dtypes
    s["success"] = pd.to_numeric(s["success"], errors="coerce").fillna(0).astype(int)
    s["fee_successful"] = pd.to_numeric(s["fee_successful"], errors="coerce").fillna(0.0)
    s["fee_not_successful"] = pd.to_numeric(s["fee_not_successful"], errors="coerce").fillna(0.0)

    # Optional sanity check: one row per transaction_id
    if ensure_unique_tx and "transaction_id" in s.columns:
        assert s["transaction_id"].is_unique, (
            "Mehrere Zeilen pro transaction_id gefunden – bitte vorher auf den ersten Versuch filtern."
        )

    # Realized fee per row (observed)
    s["fee_realized"] = s["success"]*s["fee_successful"] + (1 - s["success"])*s["fee_not_successful"]

    # KPIs
    success_rate = float(s["success"].mean())
    avg_cost = float(s["fee_realized"].mean())
    avg_fee_by_success = (
        s.groupby("success")["fee_realized"]
         .mean()
         .rename({0: "not successful", 1: "successful"})
         .to_dict()
    )
    business_score = success_rate - alpha * avg_cost

    # Print summary
    print("=== Baseline-Kennzahlen ===")
    print(f"Erfolgsrate                : {success_rate:.6f}")
    print(f"Durchschnittliche Kosten  : {avg_cost:.6f}")
    print("Ø-Gebühr nach Erfolg:")
    for k, v in avg_fee_by_success.items():
        print(f"  - {k:16s}: {v:.6f}")
    print(f"Business Score (α={alpha:.3f})        : {business_score:.6f}")

    return {
        "success_rate": success_rate,
        "avg_cost": avg_cost,
        "avg_fee_by_success": avg_fee_by_success,
        "business_score": business_score,
    }


In [7]:
df = pd.read_pickle(DF_PATH).replace([np.inf, -np.inf], np.nan).copy()

metrics = simple_baseline_metrics(df, alpha=0.05, ensure_unique_tx=True)

=== Baseline-Kennzahlen ===
Erfolgsrate                : 0.204653
Durchschnittliche Kosten  : 1.788949
Ø-Gebühr nach Erfolg:
  - not successful  : 1.232283
  - successful      : 3.952332
Business Score (α=0.050)        : 0.115206
